In [1]:
USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — PUB (12-Fold Random) Physics-Informed

## Objective
Maximize the **out-of-sample Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** under the **12-Fold Randomized Prediction in Ungauged Basins (PUB)** cross-validation protocol on the evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days).

## Data & Split Protocol
- **Spatial Regime (12-Fold PUB Cross-Validation)**:
  - `CV_FOLDS` (loaded from `pub_12fold_random_splits.json`) partitions the 531 basins into 12 mutually exclusive test folds (`~44-45` test basins and `~486-487` training basins per fold).
  - In each fold `k`, you **must** train exclusively on `fold['train_basin_mask']` using `fold['qobs_fold_train']` (`2003-10-01` to `2008-09-30`), where all test basins in `fold['test_basin_mask']` are masked to `NaN`.
  - Predict out-of-sample discharge for `fold['test_idx']` and assemble the full `Q_PRED` array of shape `(531, 3653)` across all 12 folds.
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`).
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period on each fold's training basins.
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` and `swe` across `Daymet`, `Maurer`, `NLDAS`.
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`, `HUC_02`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Hydrological Physics & Architectural Guidance
You are encouraged to integrate physical hydrological principles and domain knowledge into the model architecture, feature engineering, and training objective:
1. **Conservation Principles & Mass-Conserving Architectures**: Consider mass-conserving recurrent structures (e.g., MC-LSTM) or explicit conceptual-neural storage cells that partition precipitation and snowmelt into tracked storage states and outgoing fluxes (evapotranspiration, quickflow, interflow, slow groundwater baseflow).
2. **Snow Accumulation, Melt & Soil Moisture Accounting**: Incorporate explicit degree-day / radiation-temperature snowpack accounting (tracking snow water equivalent SWE and liquid water delivery) and multi-rate exponential Antecedent Precipitation / Soil Moisture Indices (API).
3. **Catchment Conditioning (FiLM / Hypernetworks)**: Use static catchment attributes (aridity, snow fraction, soil conductivity, geology, slope, vegetation) to modulate hidden dynamics via Feature-wise Linear Modulation (FiLM) or regularized hypernetworks.
4. **Multi-Forcing Fusion & Input Uncertainty**: Leverage the 3 meteorological forcing products (`Daymet`, `Maurer`, `NLDAS`) and auxiliary SWE/daylength inputs by learning product-uncertainty weights and cross-forcing agreement features.

In [2]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path("/tmp/kaggle_camels_era_compliant"),
    Path("/tmp/unified_benchmark"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "pub_12fold"
_splits_json = json.loads(_find_file("pub_12fold_random_splits.json").read_text(encoding="utf-8"))
CV_SPLITS = [v for _, v in sorted(_splits_json["folds"].items(), key=lambda kv: int(kv[1]["fold_index"]))]
assert len(CV_SPLITS) == 12, f"Expected 12 PUB folds, got {len(CV_SPLITS)}"

_basin_to_idx = {str(b): i for i, b in enumerate(BASINS)}
CV_FOLDS = []
_covered_test = np.zeros(len(BASINS), dtype=bool)
for _fold in CV_SPLITS:
    _fid = int(_fold["fold_index"])
    _test_idx = np.array([_basin_to_idx[str(b)] for b in _fold["test_basins"]], dtype=np.int64)
    _train_idx = np.array([_basin_to_idx[str(b)] for b in _fold["train_basins"]], dtype=np.int64)
    _test_mask = np.zeros(len(BASINS), dtype=bool)
    _test_mask[_test_idx] = True
    _train_mask = np.zeros(len(BASINS), dtype=bool)
    _train_mask[_train_idx] = True
    assert not np.any(_test_mask & _train_mask), f"Train/test overlap in PUB fold {_fid}"
    assert not np.any(_covered_test & _test_mask), f"Duplicate test basins in PUB fold {_fid}"
    _covered_test |= _test_mask
    CV_FOLDS.append({
        "fold_id": _fid,
        "train_basins": [str(b) for b in _fold["train_basins"]],
        "test_basins": [str(b) for b in _fold["test_basins"]],
        "train_idx": _train_idx,
        "test_idx": _test_idx,
        "train_basin_mask": _train_mask,
        "test_basin_mask": _test_mask,
        "qobs_fold_train": np.where(
            _train_mask[:, None] & ERA_TRAIN_MASK[None, :], QOBS_TRAIN, np.nan
        ).astype(np.float32),
    })
assert np.all(_covered_test), "12 PUB folds do not cover all 531 basins!"

print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}), {len(CV_FOLDS)} folds.")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

Loaded CAMELS-US 531 dataset (pub_12fold): 531 basins, 3653 days (1998-10-01 to 2008-09-30), 12 folds.
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [3]:
# Begin Stage

In [4]:
INSTRUCTION = [
    "To generalize to completely ungauged basins across all 12 folds, incorporate an explicit physical preprocess layer (degree-day snow accumulation/melt tracking SWE and liquid water delivery, plus multi-rate exponential Antecedent Precipitation & radiation indices) whose conservation rules hold universally in unseen catchments.",
    "Design a differentiable conceptual-neural hybrid architecture (inspired by dPL / delta-HBV / GR4J) where a regularized static-attribute network predicts bounded physical catchment parameters (soil moisture storage capacity, melt rate, baseflow recession constants) paired with a neural residual/routing network.",
    "Implement regularized Feature-wise Linear Modulation (FiLM) with static-attribute dropout and low-rank bottleneck projections so catchment climate, geology, soil permeability, and topography smoothly modulate hydrological state transitions without memorizing individual training gauges.",
    "Explore a multi-bucket water-balance routing head (partitioning effective liquid water into quick overland flow, intermediate interflow, and slow groundwater baseflow with non-negative softplus fluxes) that enforces physical runoff generation structure on unseen basins.",
    "Engineer physically meaningful dimensionless hydroclimatic ratios and energy-balance features (dynamic aridity index PET/P over 30/90/365 days, vapor pressure deficit and relative humidity estimates, rain-vs-snow phase partitioning via sigmoid temperature transition, and cross-forcing uncertainty across Daymet, Maurer, and NLDAS).",
    "Train with a physically balanced composite objective on the gauged training basins of each fold\u2014combining basin-normalized NSE, log-discharge MSE for low-flow baseflow fidelity, and water-balance bias regularization\u2014and use EMA weight averaging across epochs.",
    "Leverage multi-GPU parallel execution (`torch.cuda.device_count()`) and mixed precision (`torch.amp.autocast`) to train a multi-seed or multi-architecture physics-guided ensemble across all 12 PUB folds within the runtime budget.",
    "Synthesize the best physics-guided ungauged transfer components (universal snow/soil-moisture accounting, regularized FiLM catchment conditioning, multi-pathway routing, and EMA ensembling across all 12 folds) to maximize out-of-sample Median NSE."
]
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308

In [5]:
# [rewrite_cell]
# ==============================================================================
# Physics-Informed EA-LSTM with Forcing Uncertainty & Static Jitter for PUB
# ==============================================================================

SEQ_LEN = 365
HIDDEN_SIZE = 160
STATIC_EMB_SIZE = 64
BATCH_SIZE = 256
LR = 3e-3
EPOCHS_PER_FOLD = 40
SAMPLES_PER_BASIN = 180
EMA_ALPHA = 0.99
STATIC_NOISE_STD = 0.05

def get_hydrological_features(dyn, aux):
    """Vectorized extraction of hydrologically relevant features."""
    # Product indices: Prcp [0,5,10], Srad [1,6,11], Tmax [2,7,12], Tmin [3,8,13], Vp [4,9,14]
    # Aux: Dayl [0,2,4], SWE [1,3,5]
    
    # 1. Multi-product stats (captures forcing uncertainty)
    p_all = dyn[:, :, [0, 5, 10]]
    p_mean = np.mean(p_all, axis=-1)
    p_std = np.std(p_all, axis=-1)
    
    t_max_all = dyn[:, :, [2, 7, 12]]
    t_min_all = dyn[:, :, [3, 8, 13]]
    t_mean = (np.mean(t_max_all, axis=-1) + np.mean(t_min_all, axis=-1)) / 2.0
    t_range = np.mean(t_max_all - t_min_all, axis=-1)
    
    sr_mean = np.mean(dyn[:, :, [1, 6, 11]], axis=-1)
    vp_mean = np.mean(dyn[:, :, [4, 9, 14]], axis=-1)
    swe_mean = np.mean(aux[:, :, [1, 3, 5]], axis=-1)
    dayl_mean = np.mean(aux[:, :, [0, 2, 4]], axis=-1)

    # 2. Physics-based proxies
    # Potential Evapotranspiration (Hargreaves-Samani approx)
    pet = 0.0023 * sr_mean * (t_mean + 17.8) * np.sqrt(np.maximum(0, t_range))
    # Phase partitioning
    is_snow = (t_mean < 0).astype(np.float32)
    p_liq = p_mean * (1 - is_snow)
    p_snow = p_mean * is_snow
    # Simple melt potential
    melt_pot = np.maximum(0, t_mean * 2.0) 

    # 3. Temporal signals
    doy = pd.to_datetime(DATES).dayofyear.values
    sn = np.sin(2 * np.pi * doy / 365.25)
    cs = np.cos(2 * np.pi * doy / 365.25)

    feats = [
        np.log1p(p_mean), p_mean, p_std,
        t_mean, t_range, vp_mean,
        pet, p_liq, p_snow, melt_pot,
        swe_mean, dayl_mean,
        np.tile(sn, (dyn.shape[0], 1)),
        np.tile(cs, (dyn.shape[0], 1))
    ]
    return np.stack(feats, axis=-1).astype(np.float32)

class EALSTM(nn.Module):
    def __init__(self, dyn_dim, stat_dim, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.stat_enc = nn.Sequential(
            nn.Linear(stat_dim, STATIC_EMB_SIZE),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(STATIC_EMB_SIZE, STATIC_EMB_SIZE)
        )
        # Initialize hidden and cell states from static attributes
        self.h_init = nn.Linear(STATIC_EMB_SIZE, hidden_size)
        self.c_init = nn.Linear(STATIC_EMB_SIZE, hidden_size)
        
        # Concatenate static embedding to dynamic inputs (Entity-Aware)
        self.lstm = nn.LSTM(dyn_dim + STATIC_EMB_SIZE, hidden_size, batch_first=True)
        
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Softplus()
        )
        
    def forward(self, x_d, x_s, return_seq=False):
        s_emb = self.stat_enc(x_s) # (N, STATIC_EMB_SIZE)
        
        # Dynamic context
        s_rep = s_emb.unsqueeze(1).expand(-1, x_d.size(1), -1)
        x_combined = torch.cat([x_d, s_rep], dim=-1)
        
        # Initialization
        h0 = self.h_init(s_emb).unsqueeze(0)
        c0 = self.c_init(s_emb).unsqueeze(0)
        
        out, _ = self.lstm(x_combined, (h0.contiguous(), c0.contiguous()))
        q = self.head(out).squeeze(-1)
        return q if return_seq else q[:, -1]

# Global Feature Preprocessing
X_dyn_proc = get_hydrological_features(DYNAMIC_INPUTS, AUX_FORCINGS)
X_stat_proc = STATIC_ATTRIBUTES.copy()
# Log-transform areas and other highly skewed attributes
skewed_idx = [0, 1, 2, 3, 4] # Assuming first few are area/elev related
for i in skewed_idx:
    X_stat_proc[:, i] = np.log1p(np.maximum(0, X_stat_proc[:, i]))
X_stat_proc = np.nan_to_num(X_stat_proc, nan=0.0)

Q_PRED = np.zeros((len(BASINS), len(DATES)), dtype=np.float32)

def run_fold(fold, device_id):
    dev = torch.device(f"cuda:{device_id}")
    fid = fold["fold_id"]
    tr_mask = fold["train_basin_mask"]
    
    # Regional Scaling (only using training basins)
    d_m, d_s = X_dyn_proc[tr_mask].mean(axis=(0,1)), X_dyn_proc[tr_mask].std(axis=(0,1)) + 1e-6
    s_m, s_s = X_stat_proc[tr_mask].mean(axis=0), X_stat_proc[tr_mask].std(axis=0) + 1e-6
    
    xd_g = torch.from_numpy((X_dyn_proc - d_m) / d_s).to(dev)
    xs_g = torch.from_numpy((X_stat_proc - s_m) / s_s).to(dev)
    y_g = torch.from_numpy(fold["qobs_fold_train"]).to(dev)
    
    # Standard deviation for NSE-based weighting
    basin_std = np.ones(len(BASINS), dtype=np.float32)
    for b_idx in fold["train_idx"]:
        std = np.nanstd(fold["qobs_fold_train"][b_idx, ERA_TRAIN_MASK])
        if not np.isnan(std) and std > 1e-3: basin_std[b_idx] = std
    std_g = torch.from_numpy(basin_std).to(dev)

    train_meta = []
    for b in fold["train_idx"]:
        valid_idx = np.where(ERA_TRAIN_MASK & ~np.isnan(fold["qobs_fold_train"][b]))[0]
        valid_idx = valid_idx[valid_idx >= (SEQ_LEN - 1)]
        if len(valid_idx) > 0: train_meta.append((b, valid_idx))

    model = EALSTM(X_dyn_proc.shape[-1], X_stat_proc.shape[-1], HIDDEN_SIZE).to(dev)
    ema_model = EALSTM(X_dyn_proc.shape[-1], X_stat_proc.shape[-1], HIDDEN_SIZE).to(dev)
    ema_model.load_state_dict(model.state_dict())
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, 
                                                    steps_per_epoch=1, epochs=EPOCHS_PER_FOLD)
    
    offsets = torch.arange(-SEQ_LEN + 1, 1, device=dev).unsqueeze(0)
    rng = np.random.RandomState(SEED + fid)

    for epoch in range(EPOCHS_PER_FOLD):
        model.train()
        sb, st = [], []
        for b, v_idx in train_meta:
            sel = rng.choice(v_idx, size=min(len(v_idx), SAMPLES_PER_BASIN))
            sb.append(np.full(len(sel), b)); st.append(sel)
        
        sb, st = np.concatenate(sb), np.concatenate(st)
        perm = rng.permutation(len(sb))
        
        for i in range(0, len(sb), BATCH_SIZE):
            idx = perm[i:i+BATCH_SIZE]
            bb = torch.from_numpy(sb[idx]).to(dev)
            bt = torch.from_numpy(st[idx]).to(dev)
            
            # Static Jitter for PUB generalization
            stat_in = xs_g[bb]
            if model.training:
                stat_in = stat_in + torch.randn_like(stat_in) * STATIC_NOISE_STD
            
            optimizer.zero_grad()
            yh = model(xd_g[bb.unsqueeze(1), bt.unsqueeze(1) + offsets], stat_in)
            yt = y_g[bb, bt]
            
            # Hybrid NSE/Log-MSE Loss
            mse_loss = torch.mean(((yh - yt) / (std_g[bb] + 0.1))**2)
            log_loss = torch.mean((torch.log1p(yh) - torch.log1p(yt))**2)
            loss = mse_loss + 0.2 * log_loss
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            with torch.no_grad():
                for p, ema_p in zip(model.parameters(), ema_model.parameters()):
                    ema_p.copy_(EMA_ALPHA * ema_p + (1 - EMA_ALPHA) * p)
        scheduler.step()

    ema_model.eval()
    with torch.no_grad():
        t_idx = fold["test_idx"]
        for i in range(0, len(t_idx), 16):
            batch_b = torch.from_numpy(t_idx[i:i+16]).to(dev)
            p = ema_model(xd_g[batch_b], xs_g[batch_b], return_seq=True).cpu().numpy()
            Q_PRED[t_idx[i:i+16]] = p
    return True

# Parallel Fold Execution
n_workers = max(1, NUM_GPUS)
with ThreadPoolExecutor(max_workers=n_workers) as executor:
    futures = [executor.submit(run_fold, f, i % n_workers) for i, f in enumerate(CV_FOLDS)]
    for fut in futures:
        fut.result()

In [6]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (pub_12fold | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.72287 | Mean NSE: 0.46244 | IQR: [0.52450, 0.80566] | Basins < 0: 28
Candidate metric to be maximized: 0.722873


In [7]:
# End Stage